# Хакатон: Анализ сайта СберАвтоподписки
## Notebook 1: Загрузка данных и первичный анализ

**Цель:** Загрузить датасет, изучить структуру, оценить качество данных.

### 1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Настраиваю отображение — чтобы видеть все колонки и не обрезать числа в таблицах
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Библиотеки загружены успешно')

Библиотеки загружены успешно


### 2. Загрузка данных

In [2]:
# Путь к данным
DATA_PATH = r'D:\sber-autosubscription-hackathon'

print('Загружаем ga_sessions.pkl...')
sessions = pd.read_pickle(f'{DATA_PATH}\\ga_sessions.pkl')
print(f'ga_sessions загружен: {sessions.shape[0]:,} строк, {sessions.shape[1]} колонок')

print('\nЗагружаем ga_hits.pkl (большой файл, подождите ~1-2 мин)...')
hits = pd.read_pickle(f'{DATA_PATH}\\ga_hits.pkl')
print(f'ga_hits загружен: {hits.shape[0]:,} строк, {hits.shape[1]} колонок')

Загружаем ga_sessions.pkl...
ga_sessions загружен: 1,860,042 строк, 18 колонок

Загружаем ga_hits.pkl (большой файл, подождите ~1-2 мин)...
ga_hits загружен: 15,726,470 строк, 11 колонок


### 3. Первичный осмотр ga_sessions

In [3]:
print('=' * 60)
print('GA_SESSIONS — Информация о сессиях (визитах)')
print('=' * 60)
print(f'\nРазмер: {sessions.shape}')
print(f'Памяти занимает: {sessions.memory_usage(deep=True).sum() / 1024**2:.1f} МБ')
print('\nТипы данных:')
print(sessions.dtypes)

GA_SESSIONS — Информация о сессиях (визитах)

Размер: (1860042, 18)
Памяти занимает: 1693.7 МБ

Типы данных:
session_id                  object
client_id                   object
visit_date                  object
visit_time                  object
visit_number                 int64
utm_source                  object
utm_medium                  object
utm_campaign                object
utm_adcontent               object
utm_keyword                 object
device_category             object
device_os                   object
device_brand                object
device_model                object
device_screen_resolution    object
device_browser              object
geo_country                 object
geo_city                    object
dtype: object


In [4]:
# Смотрю на первые строки — проверяю, что данные загрузились корректно
print('Первые 5 строк ga_sessions:')
sessions.head()

Первые 5 строк ga_sessions:


,session_id,client_id,visit_date,visit_time,visit_number,utm_source,utm_medium,utm_campaign,utm_adcontent,utm_keyword,device_category,device_os,device_brand,device_model,device_screen_resolution,device_browser,geo_country,geo_city
0,9055434745589932991.1637753792.1637753792,2108382700.1637753791,2021-11-24,14:36:32,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,mobile,Android,Huawei,NaN,360x720,Chrome,Russia,Zlatoust
1,905544597018549464.1636867290.1636867290,210838531.1636867288,2021-11-14,08:21:30,1,MvfHsxITijuriZxsqZqt,cpm,FTjNLDyTrXaWYgZymFkV,xhoenQgDQsgfEPYNPwKO,IGUCNvHlhfHpROGclCit,mobile,Android,Samsung,NaN,385x854,Samsung Internet,Russia,Moscow
2,9055446045651783499.1640648526.1640648526,2108385331.1640648523,2021-12-28,02:42:06,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,mobile,Android,Huawei,NaN,360x720,Chrome,Russia,Krasnoyarsk
3,9055447046360770272.1622255328.1622255328,2108385564.1622255328,2021-05-29,05:00:00,1,kjsLglQLzykiRbcDiGcD,cpc,NaN,NOBKLgtuvqYWkXQHeYWM,NaN,mobile,None,Xiaomi,NaN,393x786,Chrome,Russia,Moscow
4,9055447046360770272.1622255345.1622255345,2108385564.1622255328,2021-05-29,05:00:00,2,kjsLglQLzykiRbcDiGcD,cpc,NaN,NaN,NaN,mobile,None,Xiaomi,NaN,393x786,Chrome,Russia,Moscow


In [5]:
# Смотрю на пропуски в sessions — важно понять, что можно использовать в модели
print('Пропуски в ga_sessions:')
missing_sessions = sessions.isnull().sum()
missing_pct_sessions = (sessions.isnull().sum() / len(sessions) * 100).round(2)
missing_df_sessions = pd.DataFrame({
    'Пропусков': missing_sessions,
    'Процент (%)': missing_pct_sessions
})
missing_df_sessions[missing_df_sessions['Пропусков'] > 0].sort_values('Процент (%)', ascending=False)

Пропуски в ga_sessions:


,Пропусков,Процент (%)
device_model,1843704,99.1200
utm_keyword,1082061,58.1700
device_os,1070138,57.5300
utm_adcontent,335615,18.0400
utm_campaign,219603,11.8100
device_brand,118678,6.3800
utm_source,97,0.0100


**Что делаю с пропусками:**

- `device_model` (99% пропусков) — не использую в модели, колонка слишком разреженная.
- `utm_keyword`, `utm_adcontent`, `utm_campaign` — заполняю строкой `'unknown'`, так как отсутствие UTM-метки само по себе несёт информацию (прямые заходы без разметки).
- `device_os`, `device_brand` — аналогично заполняю `'unknown'`.
- `utm_source` (0.01% пропусков) — единичные случаи, заполняю `'unknown'`.

Удалять строки с пропусками не стану: при 1.86 млн сессий потеря строк нежелательна, а CatBoost и логистическая регрессия нормально работают со строковым `'unknown'` как категорией.

### 4. Первичный осмотр ga_hits

In [6]:
print('=' * 60)
print('GA_HITS — Информация о событиях (хитах)')
print('=' * 60)
print(f'\nРазмер: {hits.shape}')
print(f'Памяти занимает: {hits.memory_usage(deep=True).sum() / 1024**2:.1f} МБ')
print('\nТипы данных:')
print(hits.dtypes)

GA_HITS — Информация о событиях (хитах)

Размер: (15726470, 11)
Памяти занимает: 9879.3 МБ

Типы данных:
session_id         object
hit_date           object
hit_time          float64
hit_number          int64
hit_type           object
hit_referer        object
hit_page_path      object
event_category     object
event_action       object
event_label        object
event_value        object
dtype: object


In [7]:
# Смотрю на первые строки hits
print('Первые 5 строк ga_hits:')
hits.head()

Первые 5 строк ga_hits:


,session_id,hit_date,hit_time,hit_number,hit_type,hit_referer,hit_page_path,event_category,event_action,event_label,event_value
0,5639623078712724064.1640254056.1640254056,2021-12-23,597864.0000,30,event,NaN,sberauto.com/cars?utm_source_initial=google&ut...,quiz,quiz_show,NaN,None
1,7750352294969115059.1640271109.1640271109,2021-12-23,597331.0000,41,event,NaN,sberauto.com/cars/fiat?city=1&city=18&rental_c...,quiz,quiz_show,NaN,None
2,885342191847998240.1640235807.1640235807,2021-12-23,796252.0000,49,event,NaN,sberauto.com/cars/all/volkswagen/polo/e994838f...,quiz,quiz_show,NaN,None
3,142526202120934167.1640211014.1640211014,2021-12-23,934292.0000,46,event,NaN,sberauto.com/cars?utm_source_initial=yandex&ut...,quiz,quiz_show,NaN,None
4,3450086108837475701.1640265078.1640265078,2021-12-23,768741.0000,79,event,NaN,sberauto.com/cars/all/mercedes-benz/cla-klasse...,quiz,quiz_show,NaN,None


In [8]:
# Проверяю пропуски в hits — аналогично sessions
print('Пропуски в ga_hits:')
missing_hits = hits.isnull().sum()
missing_pct_hits = (hits.isnull().sum() / len(hits) * 100).round(2)
missing_df_hits = pd.DataFrame({
    'Пропусков': missing_hits,
    'Процент (%)': missing_pct_hits
})
missing_df_hits[missing_df_hits['Пропусков'] > 0].sort_values('Процент (%)', ascending=False)

Пропуски в ga_hits:


,Пропусков,Процент (%)
event_value,15726470,100.0000
hit_time,9160322,58.2500
hit_referer,6274804,39.9000
event_label,3760184,23.9100


**Что делаю с пропусками в ga_hits:**

- `event_value` (100% пропусков) — колонку не использую совсем: в ней нет ни одного значения.
- `hit_time` (58% пропусков) — для построения целевой переменной время хита не нужно, использую только `event_action`.
- `hit_referer` (40% пропусков) — не использую в модели: источник перехода уже закодирован в UTM-метках из таблицы sessions.
- `event_label` (24% пропусков) — не использую: для определения целевого действия достаточно поля `event_action`.

Из всей таблицы `ga_hits` мне нужны только два поля: `session_id` и `event_action`. По ним определяю, была ли в сессии конверсия, и создаю признак `target`.

### 5. Анализ целевых действий

In [9]:
# Определяю список целевых действий.
# Я изучил все уникальные значения event_action в датасете и отобрал события,
# которые явно соответствуют намерению пользователя оставить заявку или инициировать контакт:
#   - sub_car_claim_click / sub_car_claim_submit_click — клик и отправка заявки на авто
#   - sub_submit_success — успешная отправка любой формы подписки
#   - sub_call_number_click — клик по номеру телефона (намерение позвонить)
#   - sub_callback_submit_click — отправка заявки на обратный звонок
#   - sub_car_request_submit_click — отправка запроса на конкретный автомобиль
#   - sub_custom_question_submit_click — отправка произвольного вопроса через форму
#   - sub_open_dialog — открытие диалогового окна контакта (в текущем датасете
#                       вхождений нет, но включаю для корректной обработки в продакшне)
#
# Все остальные события (quiz_show, page_view и т.д.) — пассивное взаимодействие,
# которое НЕ означает явного желания купить/арендовать. Поэтому я их исключаю.
target_actions = [
    'sub_car_claim_click',
    'sub_car_claim_submit_click',
    'sub_open_dialog',
    'sub_custom_question_submit_click',
    'sub_call_number_click',
    'sub_callback_submit_click',
    'sub_submit_success',
    'sub_car_request_submit_click'
]

# Проверяем, какие из них есть в датасете
found_actions = hits['event_action'].isin(target_actions).sum()
print(f'Найдено целевых событий в hits: {found_actions:,}')
print('\nРаспределение целевых действий:')
print(hits[hits['event_action'].isin(target_actions)]['event_action'].value_counts())

Найдено целевых событий в hits: 79,038

Распределение целевых действий:
event_action
sub_car_claim_click                 37928
sub_submit_success                  18439
sub_car_claim_submit_click          12359
sub_call_number_click                3653
sub_callback_submit_click            3074
sub_car_request_submit_click         2966
sub_custom_question_submit_click      619
Name: count, dtype: int64


In [10]:
# Создаю целевую переменную: помечаю сессии, где было хотя бы одно целевое действие.
# Получаю уникальные session_id из hits с целевыми событиями.
target_sessions = hits[hits['event_action'].isin(target_actions)]['session_id'].unique()
print(f'Уникальных session_id с целевым действием в hits: {len(target_sessions):,}')
print(f'Всего сессий в sessions: {sessions["session_id"].nunique():,}')

# Примечание: 383 session_id из hits отсутствуют в таблице sessions — это orphaned хиты
# (технический артефакт данных). Они автоматически отфильтруются при isin() ниже.
# Поэтому фактическое число сессий с target=1 будет 39 030 (а не 39 413).
print(f'\nКонверсия по hits: {len(target_sessions) / sessions["session_id"].nunique() * 100:.2f}%')
print('(после объединения с sessions фактическая CR = 2.10% — см. ячейку ниже)')

Уникальных session_id с целевым действием в hits: 39,413
Всего сессий в sessions: 1,860,042

Конверсия по hits: 2.12%
(после объединения с sessions фактическая CR = 2.10% — см. ячейку ниже)


In [11]:
# Добавляю признак target в таблицу sessions — 1 если была конверсия, 0 иначе
sessions['target'] = sessions['session_id'].isin(target_sessions).astype(int)

print('Распределение целевой переменной:')
print(sessions['target'].value_counts())
print(f'\nБаланс классов: {sessions["target"].mean()*100:.2f}% положительных')

Распределение целевой переменной:
target
0    1821012
1      39030
Name: count, dtype: int64

Баланс классов: 2.10% положительных


In [12]:
# Сохраняю обработанный sessions с таргетом для следующего ноутбука
print('Сохраняем данные...')
sessions.to_pickle(f'{DATA_PATH}\\sessions_with_target.pkl')
print('Сохранено: sessions_with_target.pkl')
print('\nNotebook 1 завершён. Переходим к Notebook 2 (EDA)')

Сохраняем данные...
Сохранено: sessions_with_target.pkl

Notebook 1 завершён. Переходим к Notebook 2 (EDA)
